# Cleaning Data Cuaca NOAA Indonesia 2023–2024

Di tahap ini, data dari Milestone 1 mulai dibersihkan. Nilai cuaca dipisahkan dari kode kualitasnya, lalu dicek apakah ada data kosong, laporan yang sama, atau angka yang tidak masuk akal. Hasil akhirnya disimpan ke Parquet supaya bisa dipakai untuk analisis berikutnya.


In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "noaa_global_hourly"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "noaa_indonesia_2023_2024.parquet"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


In [2]:
csv_files = sorted(path for year in ("2023", "2024") for path in (RAW_ROOT / year).glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("CSV 2023–2024 tidak ditemukan. Jalankan notebook 01 terlebih dahulu.")

columns = ["STATION", "DATE", "LATITUDE", "LONGITUDE", "ELEVATION", "NAME", "REPORT_TYPE", "QUALITY_CONTROL", "WND", "CIG", "VIS", "TMP", "DEW", "SLP"]
raw = pl.concat([pl.scan_csv(path, schema_overrides={name: pl.String for name in columns}, ignore_errors=False).select(columns) for path in csv_files], how="vertical_relaxed")
print(f"File CSV: {len(csv_files)}")
print(raw.select(pl.len().alias("baris_mentah")).collect(engine="streaming"))


File CSV: 228


shape: (1, 1)
┌──────────────┐
│ baris_mentah │
│ ---          │
│ u32          │
╞══════════════╡
│ 1273803      │
└──────────────┘


## 1. Lihat dulu bentuk data mentahnya

Isi kolom `TMP`, `DEW`, `SLP`, `WND`, `VIS`, dan `CIG` ternyata masih berupa gabungan beberapa nilai yang dipisahkan koma. Jadi, datanya dibaca sebagai teks dulu. Contoh isi kolom dan jumlah tiap kode kualitas ditampilkan supaya lebih jelas bagian mana yang perlu dibersihkan.


In [3]:
print(raw.select("TMP", "DEW", "SLP", "WND", "VIS", "CIG").head(8).collect())


shape: (8, 6)
┌─────────┬─────────┬─────────┬────────────────┬──────────────┬─────────────┐
│ TMP     ┆ DEW     ┆ SLP     ┆ WND            ┆ VIS          ┆ CIG         │
│ ---     ┆ ---     ┆ ---     ┆ ---            ┆ ---          ┆ ---         │
│ str     ┆ str     ┆ str     ┆ str            ┆ str          ┆ str         │
╞═════════╪═════════╪═════════╪════════════════╪══════════════╪═════════════╡
│ +0247,1 ┆ +0222,1 ┆ 10130,1 ┆ 120,1,N,0036,1 ┆ 006000,1,9,9 ┆ 99999,9,9,N │
│ +0262,1 ┆ +0217,1 ┆ 10147,1 ┆ 080,1,N,0036,1 ┆ 009000,1,9,9 ┆ 99999,9,9,N │
│ +0289,1 ┆ +0230,1 ┆ 10126,1 ┆ 100,1,N,0051,1 ┆ 009000,1,9,9 ┆ 99999,9,9,N │
│ +0288,1 ┆ +0228,1 ┆ 10104,1 ┆ 100,1,N,0046,1 ┆ 007000,1,9,9 ┆ 99999,9,9,N │
│ +0262,1 ┆ +0223,1 ┆ 10119,1 ┆ 090,1,N,0026,1 ┆ 007000,1,9,9 ┆ 99999,9,9,N │
│ +0253,1 ┆ +0226,1 ┆ 10148,1 ┆ 080,1,N,0026,1 ┆ 007000,1,9,9 ┆ 99999,9,9,N │
│ +0238,1 ┆ +0230,1 ┆ 10147,1 ┆ 140,1,N,0026,1 ┆ 006000,1,9,9 ┆ 99999,9,9,N │
│ +0234,1 ┆ +0217,1 ┆ 10127,1 ┆ 160,1,N,0031,1 ┆ 0

In [4]:
for column in ("TMP", "DEW", "SLP"):
    profile = (
        raw.group_by(pl.col(column).str.split(",").list.get(1, null_on_oob=True).alias("quality_code"))
        .agg(pl.len().alias("rows"))
        .sort("rows", descending=True)
        .collect(engine="streaming")
    )
    print(f"Kode kualitas {column}:")
    print(profile)


Kode kualitas TMP:
shape: (5, 2)
┌──────────────┬─────────┐
│ quality_code ┆ rows    │
│ ---          ┆ ---     │
│ str          ┆ u32     │
╞══════════════╪═════════╡
│ 1            ┆ 1267767 │
│ 5            ┆ 4165    │
│ 9            ┆ 1660    │
│ 2            ┆ 210     │
│ 6            ┆ 1       │
└──────────────┴─────────┘


Kode kualitas DEW:
shape: (4, 2)
┌──────────────┬─────────┐
│ quality_code ┆ rows    │
│ ---          ┆ ---     │
│ str          ┆ u32     │
╞══════════════╪═════════╡
│ 1            ┆ 1267283 │
│ 5            ┆ 4191    │
│ 9            ┆ 2139    │
│ 2            ┆ 190     │
└──────────────┴─────────┘


Kode kualitas SLP:
shape: (4, 2)
┌──────────────┬────────┐
│ quality_code ┆ rows   │
│ ---          ┆ ---    │
│ str          ┆ u32    │
╞══════════════╪════════╡
│ 1            ┆ 642448 │
│ 9            ┆ 631144 │
│ 2            ┆ 135    │
│ 5            ┆ 76     │
└──────────────┴────────┘


In [5]:
def part(column, index):
    return pl.col(column).str.split(",").list.get(index, null_on_oob=True)


In [6]:
prepared = raw.with_columns(
    pl.col("DATE").str.to_datetime(strict=False).alias("observed_at"),
    pl.col("LATITUDE").cast(pl.Float64, strict=False).alias("latitude"),
    pl.col("LONGITUDE").cast(pl.Float64, strict=False).alias("longitude"),
    pl.when(pl.col("ELEVATION").cast(pl.Float64, strict=False).is_between(-500, 9000)).then(pl.col("ELEVATION").cast(pl.Float64, strict=False)).otherwise(None).alias("elevation_m"),
    part("TMP", 0).cast(pl.Int32, strict=False).alias("temperature_tenths"),
    part("TMP", 1).alias("temperature_quality"),
    part("DEW", 0).cast(pl.Int32, strict=False).alias("dew_point_tenths"),
    part("DEW", 1).alias("dew_point_quality"),
    part("SLP", 0).cast(pl.Int32, strict=False).alias("pressure_tenths"),
    part("SLP", 1).alias("pressure_quality"),
    part("WND", 0).cast(pl.Int32, strict=False).alias("wind_direction_raw"),
    part("WND", 1).alias("wind_direction_quality"),
    part("WND", 3).cast(pl.Int32, strict=False).alias("wind_speed_tenths"),
    part("WND", 4).alias("wind_speed_quality"),
    part("VIS", 0).cast(pl.Int32, strict=False).alias("visibility_raw"),
    part("VIS", 1).alias("visibility_quality"),
    part("CIG", 0).cast(pl.Int32, strict=False).alias("ceiling_raw"),
    part("CIG", 1).alias("ceiling_quality"),
)


In [7]:
checks = prepared.select(
    pl.len().alias("raw_rows"),
    (pl.col("STATION").is_null() | (pl.col("STATION") == "") | pl.col("observed_at").is_null()).sum().alias("missing_key_rows"),
    pl.struct(columns).is_duplicated().sum().alias("exact_duplicate_rows"),
    pl.col("temperature_tenths").eq(9999).sum().alias("temperature_sentinel_rows"),
    pl.col("wind_speed_tenths").eq(9999).sum().alias("wind_sentinel_rows"),
    pl.col("visibility_raw").eq(999999).sum().alias("visibility_sentinel_rows"),
).collect(engine="streaming")
print(checks)


shape: (1, 6)
┌──────────┬─────────────────┬─────────────────┬─────────────────┬────────────────┬────────────────┐
│ raw_rows ┆ missing_key_row ┆ exact_duplicate ┆ temperature_sen ┆ wind_sentinel_ ┆ visibility_sen │
│ ---      ┆ s               ┆ _rows           ┆ tinel_rows      ┆ rows           ┆ tinel_rows     │
│ u32      ┆ ---             ┆ ---             ┆ ---             ┆ ---            ┆ ---            │
│          ┆ u32             ┆ u32             ┆ u32             ┆ u32            ┆ u32            │
╞══════════╪═════════════════╪═════════════════╪═════════════════╪════════════════╪════════════════╡
│ 1273803  ┆ 0               ┆ 0               ┆ 1660            ┆ 3068           ┆ 2264           │
└──────────┴─────────────────┴─────────────────┴─────────────────┴────────────────┴────────────────┘


## 2. Pisahkan nilai cuaca dan kode kualitasnya

Ada angka khusus yang menandakan data tidak tersedia. Contohnya `+9999` pada suhu dan titik embun, `99999` pada tekanan, serta `999999` pada jarak pandang. Angka seperti ini disebut sentinel dan tidak boleh ikut dihitung sebagai pengukuran cuaca.

Kode kualitas `9` juga perlu diperhatikan. Pada elemen utama NOAA, kode ini bisa berarti nilainya ada dan sudah lolos pengecekan batas kasar. Di revisi awal, kode ini sempat dikeluarkan. Setelah dicocokkan dengan dokumentasi NOAA, aturannya diperbaiki: kode `9` boleh dipakai selama angkanya bukan sentinel dan masih masuk rentang yang dipilih.


In [8]:
quality_counts = prepared.select(
    pl.len().alias("rows_before"),
    pl.col("STATION").n_unique().alias("stations_before"),
    (~pl.col("latitude").is_between(-11.5, 6.5) | ~pl.col("longitude").is_between(94.0, 142.0)).sum().alias("outside_coordinate_range"),
    pl.col("elevation_m").is_null().sum().alias("invalid_elevation"),
    pl.col("temperature_quality").eq("9").sum().alias("temperature_quality_9"),
    pl.col("pressure_quality").eq("9").sum().alias("pressure_quality_9"),
).collect(engine="streaming")
print(quality_counts)

shape: (1, 6)
┌─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ rows_before ┆ stations_befor ┆ outside_coordi ┆ invalid_elevat ┆ temperature_qu ┆ pressure_quali │
│ ---         ┆ e              ┆ nate_range     ┆ ion            ┆ ality_9        ┆ ty_9           │
│ u32         ┆ ---            ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│             ┆ u32            ┆ u32            ┆ u32            ┆ u32            ┆ u32            │
╞═════════════╪════════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ 1273803     ┆ 114            ┆ 0              ┆ 5555           ┆ 1660           ┆ 631144         │
└─────────────┴────────────────┴────────────────┴────────────────┴────────────────┴────────────────┘


## 3. Mulai bersihkan datanya

Baris tanpa ID stasiun atau waktu yang valid dikeluarkan. Waktu di luar 2023–2024 dan koordinat di luar batas yang dipakai juga disaring. Kalau ada baris yang sama persis pada 14 kolom inti, cukup disimpan satu.

Laporan dari stasiun dan waktu yang sama belum tentu duplikat karena jenis atau isinya bisa berbeda. Jadi, laporan seperti itu tetap disimpan. Kalau hanya satu pengukuran cuaca yang bermasalah, nilainya dibuat `null` supaya pengukuran lain pada baris tersebut masih bisa digunakan.


In [9]:
valid = prepared.filter(
    pl.col("STATION").is_not_null()
    & (pl.col("STATION") != "")
    & pl.col("observed_at").is_not_null()
    & pl.col("observed_at").dt.year().is_in([2023, 2024])
    & pl.col("latitude").is_between(-11.5, 6.5)
    & pl.col("longitude").is_between(94.0, 142.0)
).unique(subset=columns, keep="first", maintain_order=False)


In [10]:
clean = valid.with_columns(
    pl.when(pl.col("temperature_tenths").is_between(-600, 600) & pl.col("temperature_quality").is_in(["0", "1", "4", "5", "9"]))
      .then(pl.col("temperature_tenths") / 10).otherwise(None).alias("temperature_c"),
    pl.when(pl.col("dew_point_tenths").is_between(-600, 600) & pl.col("dew_point_quality").is_in(["0", "1", "4", "5", "9"]))
      .then(pl.col("dew_point_tenths") / 10).otherwise(None).alias("dew_point_c"),
    pl.when(pl.col("pressure_tenths").is_between(8700, 10850) & pl.col("pressure_quality").is_in(["0", "1", "4", "5", "9"]))
      .then(pl.col("pressure_tenths") / 10).otherwise(None).alias("sea_level_pressure_hpa"),
    pl.when(pl.col("wind_direction_raw").is_between(0, 360) & pl.col("wind_direction_quality").is_in(["0", "1", "4", "5", "9"]))
      .then(pl.col("wind_direction_raw")).otherwise(None).alias("wind_direction_deg"),
    pl.when(pl.col("wind_speed_tenths").is_between(0, 750) & pl.col("wind_speed_quality").is_in(["0", "1", "4", "5", "9"]))
      .then(pl.col("wind_speed_tenths") / 10).otherwise(None).alias("wind_speed_ms"),
    pl.when(pl.col("visibility_raw").is_between(0, 160000) & pl.col("visibility_quality").is_in(["0", "1", "4", "5", "9"]))
      .then(pl.col("visibility_raw")).otherwise(None).alias("visibility_m"),
    pl.when(pl.col("ceiling_raw").is_between(0, 20000) & pl.col("ceiling_quality").is_in(["0", "1", "4", "5", "9"]))
      .then(pl.col("ceiling_raw")).otherwise(None).alias("ceiling_m"),
).select(
    pl.col("STATION").alias("station_id"), "observed_at",
    "latitude", "longitude", "elevation_m",
    pl.col("NAME").alias("station_name"),
    pl.col("REPORT_TYPE").alias("report_type"),
    pl.col("QUALITY_CONTROL").alias("quality_control"),
    "temperature_c", "temperature_quality", "dew_point_c", "dew_point_quality",
    "sea_level_pressure_hpa", "pressure_quality", "wind_direction_deg",
    "wind_direction_quality", "wind_speed_ms", "wind_speed_quality",
    "visibility_m", "visibility_quality", "ceiling_m", "ceiling_quality",
)


In [11]:
clean.sink_parquet(OUTPUT_PATH)
print(f"Parquet: {OUTPUT_PATH}")
print(f"Ukuran: {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MiB")


Parquet: /home/jovyan/work/data/processed/noaa_indonesia_2023_2024.parquet
Ukuran: 19.02 MiB


## 4. Cek lagi hasilnya dengan DuckDB

Setelah disimpan, hasil cleaning dibuka lagi lewat DuckDB. `SUMMARIZE` dipakai untuk melihat rentang nilai dan persentase data kosong. Jumlah stasiun, periode pengamatan, dan laporan pada waktu yang sama juga dicek supaya hasilnya bisa dibandingkan dengan data awal.


In [12]:
con = duckdb.connect()
path_sql = OUTPUT_PATH.as_posix().replace("'", "''")
print(con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{path_sql}')").df().to_string(index=False))


           column_name column_type                          min                   max  approx_unique                        avg                std                        q25                        q50                        q75   count  null_percentage
            station_id     VARCHAR                  96001099999           97980099999            128                        NaN                NaN                        NaN                        NaN                        NaN 1273803             0.00
           observed_at   TIMESTAMP          2023-01-01 00:00:00   2024-12-31 23:30:00          41962 2024-01-17 03:15:57.286488                NaN 2023-07-13 16:54:08.877993 2024-01-19 18:44:46.767129 2024-07-30 15:28:03.352993 1273803             0.00
              latitude      DOUBLE                  -10.7333333              5.874131             98        -3.3378509697117487 3.9845732293067497         -6.713814968333382        -3.4173743648553336        -0.3417100605030218 1273803      

In [13]:
quality = con.execute(f'''SELECT COUNT(*) AS clean_rows,
COUNT(DISTINCT station_id) AS stations,
MIN(observed_at) AS first_observation,
MAX(observed_at) AS last_observation,
COUNT(*) - COUNT(temperature_c) AS missing_temperature,
COUNT(*) - COUNT(dew_point_c) AS missing_dew_point,
COUNT(*) - COUNT(sea_level_pressure_hpa) AS missing_pressure,
COUNT(*) - COUNT(wind_speed_ms) AS missing_wind_speed,
COUNT(*) - COUNT(visibility_m) AS missing_visibility,
COUNT(*) - COUNT(ceiling_m) AS missing_ceiling
FROM read_parquet('{path_sql}')''').df()
print(quality.to_string(index=False))


 clean_rows  stations first_observation    last_observation  missing_temperature  missing_dew_point  missing_pressure  missing_wind_speed  missing_visibility  missing_ceiling
    1273803       114        2023-01-01 2024-12-31 23:30:00                 1871               2329            631279                3217                2264           803349


In [14]:
duplicate_keys = con.execute(f'''SELECT COUNT(*) AS duplicate_station_time_groups FROM (
SELECT station_id, observed_at FROM read_parquet('{path_sql}')
GROUP BY station_id, observed_at HAVING COUNT(*) > 1)''').fetchone()[0]
print(f"Grup stasiun-waktu dengan lebih dari satu laporan: {duplicate_keys}")


Grup stasiun-waktu dengan lebih dari satu laporan: 102255


## 5. Bandingkan aturan dan cek beberapa hal lagi

Untuk suhu, ada tiga cara yang dibandingkan: hanya mengecek rentang angka, memakai kode kualitas yang lebih ketat, dan memakai aturan cleaning yang dipilih. Dari selisih jumlahnya, bisa terlihat pengaruh pengecekan kualitas terhadap data yang dipakai.

Nama stasiun yang sama juga dicek karena bisa saja dipakai oleh lebih dari satu ID. ID stasiun tetap dijadikan patokan. Setelah itu, jumlah baris, persentase nilai kosong, dan hubungan titik embun dengan suhu diperiksa lagi.


In [15]:
quality_comparison = prepared.select(
    pl.len().alias("rows"),
    pl.col("temperature_tenths").is_between(-600, 600).sum().alias("range_only"),
    (pl.col("temperature_tenths").is_between(-600, 600) & pl.col("temperature_quality").is_in(["1", "5"])).sum().alias("strict_quality"),
    (pl.col("temperature_tenths").is_between(-600, 600) & pl.col("temperature_quality").is_in(["0", "1", "4", "5", "9"])).sum().alias("selected_quality"),
).collect(engine="streaming")
print(quality_comparison)

shape: (1, 4)
┌─────────┬────────────┬────────────────┬──────────────────┐
│ rows    ┆ range_only ┆ strict_quality ┆ selected_quality │
│ ---     ┆ ---        ┆ ---            ┆ ---              │
│ u32     ┆ u32        ┆ u32            ┆ u32              │
╞═════════╪════════════╪════════════════╪══════════════════╡
│ 1273803 ┆ 1272143    ┆ 1271932        ┆ 1271932          │
└─────────┴────────────┴────────────────┴──────────────────┘


In [16]:
station_mapping = con.execute(f"""SELECT station_name, COUNT(DISTINCT station_id) AS station_ids,
string_agg(DISTINCT station_id, ', ') AS ids
FROM read_parquet('{path_sql}') GROUP BY station_name
HAVING COUNT(DISTINCT station_id) > 1 ORDER BY station_name""").df()
print(station_mapping.to_string(index=False))

station_name  station_ids                      ids
 EL TARI, ID            2 97372099999, 97372599999


In [17]:
same_report_groups = con.execute(f"""SELECT report_type, COUNT(*) AS repeated_groups FROM (
SELECT station_id, observed_at, report_type, COUNT(*) AS reports
FROM read_parquet('{path_sql}') GROUP BY station_id, observed_at, report_type
HAVING COUNT(*) > 1) GROUP BY report_type ORDER BY repeated_groups DESC""").df()
print(same_report_groups.to_string(index=False))

Empty DataFrame
Columns: [report_type, repeated_groups]
Index: []


In [18]:
clean_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path_sql}')").fetchone()[0]
raw_rows = checks["raw_rows"][0]
print(pl.DataFrame({"stage": ["raw", "clean", "removed"], "rows": [raw_rows, clean_rows, raw_rows - clean_rows]}))
assert 0 < clean_rows <= raw_rows

shape: (3, 2)
┌─────────┬─────────┐
│ stage   ┆ rows    │
│ ---     ┆ ---     │
│ str     ┆ i64     │
╞═════════╪═════════╡
│ raw     ┆ 1273803 │
│ clean   ┆ 1273803 │
│ removed ┆ 0       │
└─────────┴─────────┘


In [19]:
measurements = ["temperature_c", "dew_point_c", "sea_level_pressure_hpa", "wind_speed_ms", "visibility_m", "ceiling_m"]
missing_rows = []
for column in measurements:
    total, missing = con.execute(f"SELECT COUNT(*), COUNT(*) - COUNT({column}) FROM read_parquet('{path_sql}')").fetchone()
    missing_rows.append({"variable": column, "missing": missing, "missing_pct": round(missing / total * 100, 2)})
print(pl.DataFrame(missing_rows).sort("missing_pct", descending=True))

shape: (6, 3)
┌────────────────────────┬─────────┬─────────────┐
│ variable               ┆ missing ┆ missing_pct │
│ ---                    ┆ ---     ┆ ---         │
│ str                    ┆ i64     ┆ f64         │
╞════════════════════════╪═════════╪═════════════╡
│ ceiling_m              ┆ 803349  ┆ 63.07       │
│ sea_level_pressure_hpa ┆ 631279  ┆ 49.56       │
│ wind_speed_ms          ┆ 3217    ┆ 0.25        │
│ dew_point_c            ┆ 2329    ┆ 0.18        │
│ visibility_m           ┆ 2264    ┆ 0.18        │
│ temperature_c          ┆ 1871    ┆ 0.15        │
└────────────────────────┴─────────┴─────────────┘


In [20]:
physical_checks = con.execute(f"""SELECT
COUNT(*) FILTER (WHERE dew_point_c > temperature_c) AS dew_above_temperature,
COUNT(*) FILTER (WHERE wind_speed_ms < 0) AS negative_wind,
COUNT(*) FILTER (WHERE station_id IS NULL OR observed_at IS NULL) AS missing_keys,
COUNT(*) FILTER (WHERE elevation_m IS NULL) AS missing_elevation
FROM read_parquet('{path_sql}')""").df()
print(physical_checks.to_string(index=False))
assert int(physical_checks.loc[0, "negative_wind"]) == 0
assert int(physical_checks.loc[0, "missing_keys"]) == 0

 dew_above_temperature  negative_wind  missing_keys  missing_elevation
                     0              0             0               5555


## Catatan dari hasil cleaning

- Jumlah laporan tetap 1.273.803 baris. Pada data ini, tidak ada baris yang terbuang; cleaning terutama mengubah nilai pengukuran yang bermasalah menjadi `null`.
- Ada 211 pengukuran suhu yang angkanya masih masuk rentang, tetapi tidak lolos aturan kualitas yang dipakai.
- Nama `EL TARI, ID` dipakai oleh dua ID stasiun. Keduanya tetap dibedakan berdasarkan ID.
- Tidak ditemukan laporan berulang dengan kombinasi ID stasiun, waktu, dan jenis laporan yang sama.
- Data kosong paling banyak ada pada ketinggian awan (63,07%) dan tekanan udara (49,56%). Ini perlu diperhatikan saat memilih variabel untuk EDA.
- Suhu dan titik embun dibagi 10 menjadi derajat Celsius. Tekanan dan kecepatan angin juga dibagi 10 agar sesuai satuannya.
- Batas koordinat yang dipakai hanya untuk pengecekan awal wilayah Indonesia, bukan batas wilayah yang detail. Batas nilai cuaca juga merupakan aturan analisis yang masih perlu ditinjau sesuai kebutuhan.
- File Parquet disimpan di folder lokal dan tidak ikut diunggah ke GitHub.

Acuan format dan kode kualitas: [dokumentasi NOAA ISD](https://www.ncei.noaa.gov/pub/data/noaa/isd-format-document.pdf).
